In [2]:
!pip install -q wandb omegaconf timm albumentations tqdm python-dotenv rich scipy scikit-learn

import os
os.environ['WANDB_API_KEY'] = 'WANDB_API_KEY'  # ← THAY KEY CỦA BẠN

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.9.0+cu126
CUDA: True
GPU: Tesla T4


In [3]:
import shutil, os
from pathlib import Path

# === Auto-detect dataset path ===
# Kaggle đường dẫn có thể là /kaggle/input/holmhz-data-v3
# hoặc /kaggle/input/<username>/holmhz-data-v3
# → tự tìm folder chứa "data" và "src"
DATA_INPUT = None
for p in Path("/kaggle/input").rglob("src"):
    if p.is_dir() and (p.parent / "data").exists():
        DATA_INPUT = p.parent
        break

# Fallback: nếu không tìm được, list tất cả để debug
if DATA_INPUT is None:
    print("❌ Không tìm được dataset! Listing /kaggle/input:")
    for item in sorted(Path("/kaggle/input").rglob("*")):
        if item.is_dir() and len(item.parts) <= 6:
            print(f"  {item}")
    raise FileNotFoundError("Dataset not found. Kiểm tra dataset đã Add đúng chưa.")

print(f"✅ Found dataset at: {DATA_INPUT}")
os.system(f'ls "{DATA_INPUT}"')

# --- Copy code (xóa cũ nếu có → copy mới) ---
for d in ["src", "scripts", "preprocessing", "configs"]:
    src, dst = DATA_INPUT / d, Path(d)
    if not src.exists():
        print(f"  ⚠️ WARNING: {src} not found in dataset!"); continue
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"Copied {d}/")

for f in ["pyproject.toml"]:
    src, dst = DATA_INPUT / f, Path(f)
    if src.exists():
        shutil.copy2(src, dst); print(f"Copied {f}")

# --- Copy data (xóa cũ nếu có → copy mới) ---
for folder in ["processed", "manifests"]:
    src = DATA_INPUT / "data" / folder
    dst = Path(f"data/{folder}")
    if not src.exists():
        print(f"  ⚠️ WARNING: {src} not found!"); continue
    if dst.exists():
        shutil.rmtree(dst)
    print(f"Copying data/{folder} ...")
    shutil.copytree(src, dst)
    print(f"  Done!")

# --- Verify ---
print("\n=== Data structure ===")
!find data/processed -maxdepth 3 -type d
print()
!echo "OOD tristanzhang_fake:" && ls data/processed/ood_test/tristanzhang_fake/ | wc -l
!echo "Train tristanzhang_train:" && ls data/processed/train/fake_diffusion/tristanzhang_train/ | wc -l
!echo "Manifests:" && ls data/manifests/

✅ Found dataset at: /kaggle/input/datasets/eurusdevsec/holmhz-data-v3
configs
data
preprocessing
pyproject.toml
scripts
src
Copied src/
Copied scripts/
Copied preprocessing/
Copied configs/
Copied pyproject.toml
Copying data/processed ...
  Done!
Copying data/manifests ...
  Done!

=== Data structure ===
data/processed
data/processed/train
data/processed/train/real
data/processed/train/real/real_camera_train
data/processed/train/real/ffhq
data/processed/train/real/real_pexels_train
data/processed/train/real/diverse_real
data/processed/train/real/cifake
data/processed/train/fake_diffusion
data/processed/train/fake_diffusion/sd15
data/processed/train/fake_diffusion/cifake
data/processed/train/fake_diffusion/tristanzhang_train
data/processed/train/fake_gan
data/processed/train/fake_gan/stylegan
data/processed/ood_test
data/processed/ood_test/real_pexels
data/processed/ood_test/tristanzhang_fake
data/processed/ood_test/flux
data/processed/ood_test/real_camera

OOD tristanzhang_fake:
500
Tr

In [4]:
import random, shutil, json
from pathlib import Path

OOD_DIR   = Path("data/processed/ood_test/tristanzhang_fake")  # 500 ảnh
TRAIN_DIR = Path("data/processed/train/fake_diffusion/tristanzhang_train")
MANIFESTS = Path("data/manifests")
SEED = 42

# --- Step 1: Copy ảnh vào tristanzhang_train nếu chưa có ---
existing = list(TRAIN_DIR.glob("*.png")) if TRAIN_DIR.exists() else []
if len(existing) == 0:
    print("=== Creating tristanzhang_train ===")
    TRAIN_DIR.mkdir(parents=True, exist_ok=True)

    all_imgs = sorted(OOD_DIR.glob("*.png"))
    print(f"  Source: {len(all_imgs)} images in OOD tristanzhang_fake")
    assert len(all_imgs) >= 200, f"Not enough source images: {len(all_imgs)}"

    random.seed(SEED)
    random.shuffle(all_imgs)
    for img in all_imgs[:200]:
        shutil.copy2(img, TRAIN_DIR / img.name)
    print(f"  Copied 200 images → {TRAIN_DIR}")
else:
    print(f"tristanzhang_train already exists: {len(existing)} images")

# --- Step 2: Luôn tái tạo filter file từ nội dung thực tế của TRAIN_DIR ---
# (tránh lỗi nếu filter file cũ không khớp với ảnh đang có trong TRAIN_DIR)
MANIFESTS.mkdir(parents=True, exist_ok=True)
train_stems  = {p.stem for p in TRAIN_DIR.glob("*.png")}
all_ood_imgs = sorted(OOD_DIR.glob("*.png"))
test_imgs    = [p for p in all_ood_imgs if p.stem not in train_stems]

filter_path = MANIFESTS / "tristanzhang_test_only.txt"
with open(filter_path, "w") as f:
    for img in test_imgs:
        f.write(img.stem + ".jpg\n")

print(f"Filter written: {len(train_stems)} train | {len(test_imgs)} test  →  {filter_path}")
assert len(train_stems) + len(test_imgs) == len(all_ood_imgs), \
    f"Split không đủ: {len(train_stems)} + {len(test_imgs)} != {len(all_ood_imgs)}"

# --- Step 3: Rebuild manifests ---
print("\n=== Rebuilding manifests ===")
import subprocess
result = subprocess.run(["python", "preprocessing/build_splits.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("❌ build_splits.py FAILED!")
    print(result.stderr)
    raise RuntimeError("build_splits.py failed — xem lỗi ở trên")

# --- Step 4: Verify ---
manifest_path = Path("data/manifests/train.json")
assert manifest_path.exists(), f"Manifest không tồn tại: {manifest_path}"

train_data = json.load(open("data/manifests/train.json"))
ood_data   = json.load(open("data/manifests/test_ood.json"))

train_sources = {}
for x in train_data:
    train_sources[x["source"]] = train_sources.get(x["source"], 0) + 1

print(f"Train: {len(train_data)} total")
for s, c in sorted(train_sources.items()):
    print(f"  {s:32s} {c:5d}")

ood_sources = {}
for x in ood_data:
    ood_sources[x["source"]] = ood_sources.get(x["source"], 0) + 1
print(f"\nOOD: {len(ood_data)} total")
for s, c in sorted(ood_sources.items()):
    print(f"  {s:32s} {c:5d}")

# Checks
ok = True
if "tristanzhang_train" not in train_sources:
    print("\n❌ tristanzhang_train KHÔNG có trong training data!"); ok = False
if len(train_data) < 20000:
    print(f"\n❌ Train size quá nhỏ: {len(train_data)}"); ok = False
if len(ood_data) < 600:
    print(f"\n❌ OOD size quá nhỏ: {len(ood_data)}"); ok = False
if ok:
    print(f"\n✅ ALL CHECKS PASSED — sẵn sàng train!")

tristanzhang_train already exists: 200 images
Filter written: 200 train | 300 test  →  data/manifests/tristanzhang_test_only.txt

=== Rebuilding manifests ===
BUILD SPLITS — HolmHz Data Pipeline

📂 Scanning training data...
  ✅ real/cifake: 7000 ảnh (label=0)
  ✅ real/diverse_real: 3000 ảnh (label=0)
  ✅ real/ffhq: 5000 ảnh (label=0)
  ✅ real/real_camera_train: 300 ảnh (label=0)
  ✅ real/real_pexels_train: 300 ảnh (label=0)
  ✅ fake_gan/stylegan: 5000 ảnh (label=1)
  ✅ fake_diffusion/cifake: 7000 ảnh (label=1)
  ✅ fake_diffusion/sd15: 2500 ảnh (label=1)
  ✅ fake_diffusion/tristanzhang_train: 200 ảnh (label=1)

📊 Tổng cộng training data: 30300 ảnh
   Real: 15600
   Fake: 14700

✂️  Chia stratified (70%/15%/15%, seed=42)...

📂 Scanning OOD test data...
  📋 Loaded real_pexels test-only filter: 200 files
  📋 Loaded tristanzhang test-only filter: 300 files
  ✅ ood_test/flux: 80 ảnh (label=1 → fake)
  ✅ ood_test/real_camera: 100 ảnh (label=0 → real)
  ✅ ood_test/real_pexels: 200 ảnh (label=0

In [6]:
# Thêm src/ vào Python path (thay vì pip install -e . vì hatchling lỗi trên Kaggle)
import sys, os
sys.path.insert(0, "src")
os.environ["PYTHONPATH"] = "src"  # Để scripts/train.py cũng tìm được holmhz

import holmhz
print(f"HolmHz: {holmhz.__version__}")

HolmHz: 0.1.0


In [9]:
# ═══ Cell 5: Write config v5 ═══
config_v5 = """
model:
  name: efficientnet_b0
  pretrained: true
  num_classes: 1
  dropout: 0.3
  freeze_backbone: false

training:
  epochs: 30
  batch_size: 32
  learning_rate: 0.0001
  optimizer: adamw
  weight_decay: 0.0001
  scheduler: cosine
  pos_weight: 1.0
  early_stopping:
    patience: 10
    monitor: val_auc

data:
  train_manifest: data/manifests/train.json
  val_manifest: data/manifests/val.json
  image_size: 224
  num_workers: 4
  augmentation: true
  use_weighted_sampler: true

wandb:
  project: holmhz
  entity: null
  log_every_n_steps: 10
""".strip()

with open("configs/train_v5.yaml", "w") as f:
    f.write(config_v5)
print("✅ configs/train_v5.yaml written")

✅ configs/train_v5.yaml written


In [10]:
# ═══ Cell 6: Train ═══
# Xóa old checkpoints
import os, shutil
for f in ["outputs/checkpoints/best.pt", "outputs/checkpoints/last.pt"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"🗑️ Removed {f}")

os.makedirs("outputs/checkpoints", exist_ok=True)

# Verify data
import json
with open("data/manifests/train.json") as f:
    td = json.load(f)
sources = {}
for e in td:
    sources[e["source"]] = sources.get(e["source"], 0) + 1
print(f"\nTrain: {len(td)} samples")
for s, c in sorted(sources.items()):
    print(f"  {s:30s} {c}")

assert "real_camera_train" in sources, "❌ real_camera_train MISSING!"
assert "tristanzhang_train" in sources, "❌ tristanzhang_train MISSING!"
print("\n✅ ALL CHECKS PASSED — Training...")

# Train
!PYTHONPATH=src python scripts/train.py configs/train_v5.yaml data.num_workers=4


Train: 21210 samples
  cifake                         9800
  diverse_real                   2100
  ffhq                           3500
  real_camera_train              210
  real_pexels_train              210
  sd15                           1750
  stylegan                       3500
  tristanzhang_train             140

✅ ALL CHECKS PASSED — Training...
2026-03-03 06:54:36 | INFO     | Config loaded from: configs/train_v5.yaml
2026-03-03 06:54:36 | INFO     | Config:
model:
  name: efficientnet_b0
  pretrained: true
  num_classes: 1
  dropout: 0.3
  freeze_backbone: false
training:
  epochs: 30
  batch_size: 32
  learning_rate: 0.0001
  optimizer: adamw
  weight_decay: 0.0001
  scheduler: cosine
  pos_weight: 1.0
  early_stopping:
    patience: 10
    monitor: val_auc
data:
  train_manifest: data/manifests/train.json
  val_manifest: data/manifests/val.json
  image_size: 224
  num_workers: 4
  augmentation: true
  use_weighted_sampler: true
wandb:
  project: holmhz
  entity: null
  lo

In [11]:
# ═══ Cell 7: Save results ═══
import shutil
from pathlib import Path

best = Path("outputs/checkpoints/best.pt")
if best.exists():
    shutil.copy2(best, "/kaggle/working/best_v5.pt")
    print(f"✅ Saved /kaggle/working/best_v5.pt")
else:
    print("⚠️ best.pt not found!")

last = Path("outputs/checkpoints/last.pt")
if last.exists():
    shutil.copy2(last, "/kaggle/working/last_v5.pt")

# Quick eval
!PYTHONPATH=src python scripts/test.py model.checkpoint=outputs/checkpoints/best.pt \
    data.num_workers=4 data.batch_size=32 evaluation.threshold=0.76

✅ Saved /kaggle/working/best_v5.pt
2026-03-03 07:42:33 | INFO     | Config: configs/test.yaml
2026-03-03 07:42:33 | INFO     | Device: cuda
2026-03-03 07:42:34 | INFO     | Loaded: outputs/checkpoints/best.pt (epoch 24, val_auc 0.9971778433291415)
2026-03-03 07:42:34 | INFO     | ID test:  4545 samples
2026-03-03 07:42:34 | INFO     | OOD test: 680 samples

IN-DOMAIN EVALUATION
2026-03-03 07:42:34 | INFO     | Evaluating 4545 samples (143 batches)...
2026-03-03 07:42:41 | INFO     | Overall — AUC: 0.9961, Acc: 0.9732, F1: 0.9721 
2026-03-03 07:42:41 | INFO     |   cifake               — Acc: 0.9629, N: 2100
2026-03-03 07:42:41 | INFO     |   diverse_real         — Acc: 0.9889, N: 450
2026-03-03 07:42:41 | INFO     |   ffhq                 — Acc: 0.9987, N: 750
2026-03-03 07:42:41 | INFO     |   real_camera_train    — Acc: 0.9333, N: 45
2026-03-03 07:42:41 | INFO     |   real_pexels_train    — Acc: 0.8444, N: 45
2026-03-03 07:42:41 | INFO     |   sd15                 — Acc: 0.9867, N: 3